# 06 — Ground Truth

**Tujuan:** membangun ground truth label (match / non-match) untuk evaluasi independen di Notebook 07,
dengan menggabungkan silver-standard otomatis (dari `customer_id`) dan sampel manual review.

**Kenapa ini perlu (bukan cuma pakai silver-standard saja):**
- Silver-standard (`customer_id` sama) HANYA berisi positive label (match) — Notebook 04 sudah
  membuktikan ini menyebabkan data leakage dan probability saturated saat dipakai untuk training DAN evaluasi.
- Tidak ada genuine hard-negative (pasangan yang mirip tapi BUKAN orang yang sama) di candidate set manapun sejauh ini.
- Rule 6 (Avoid data leakage) mengharuskan data tuning dan evaluasi dipisah — ground truth di sini akan
  displit jadi tuning set dan holdout set SEBELUM dipakai di Notebook 07.

**Yang dihasilkan notebook ini:**
1. `review_queue.csv` — pairs yang perlu di-review MANUAL oleh manusia (kolom label dikosongkan, diisi di luar notebook)
2. `ground_truth_labels.csv` — gabungan silver-standard + hasil manual review (setelah `review_queue.csv` diisi), displit tuning/holdout

**PENTING:** Section manual review MEMBUTUHKAN INPUT MANUSIA. Notebook ini tidak bisa dijalankan
end-to-end tanpa jeda — setelah `review_queue.csv` diekspor, Anda perlu mengisi kolom `manual_label`
di file tersebut (Excel/spreadsheet), baru lanjut ke section terakhir.

## 1. Imports & paths

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

STANDARDIZED_PATH = r"C:\Users\User\Downloads\Fix\data\raw\customers_standardized.csv"
DECISION_PATH     = r"C:\Users\User\Downloads\Fix\data\raw\decided_matches.csv"
CLUSTER_ANALYSIS_PATH = r"C:\Users\User\Downloads\Fix\data\raw\cluster_analysis.csv"

REVIEW_QUEUE_PATH   = r"C:\Users\User\Downloads\Fix\data\raw\review_queue.csv"
GROUND_TRUTH_PATH   = r"C:\Users\User\Downloads\Fix\data\raw\ground_truth_labels.csv"

RANDOM_SEED = 42

## 2. Load data

In [ ]:
df       = pd.read_csv(STANDARDIZED_PATH, dtype=str)
pred     = pd.read_csv(DECISION_PATH, dtype=str)
clusters = pd.read_csv(CLUSTER_ANALYSIS_PATH, dtype=str)

pred['match_probability'] = pred['match_probability'].astype(float)
clusters['cluster_size']  = clusters['cluster_size'].astype(int)

print(f"Standardized data : {len(df):,} rows")
print(f"Decided pairs     : {len(pred):,} rows")
print(f"Clusters          : {len(clusters):,} rows, {clusters['cluster_id'].nunique():,} unique cluster_id")

## 3. Silver-standard labels (otomatis, dari `customer_id`)

**Batasan yang harus diingat (Rule 2 — jangan menganggap ini ground truth sempurna):**
- Label ini **hanya positive** (pasangan dengan `customer_id` sama = diasumsikan match).
- Ini asumsi yang kuat tapi TIDAK 100% valid — kita belum pernah memverifikasi manual bahwa
  `customer_id` yang sama selalu berarti orang yang sama (walau dari Notebook 01, pola typo nama
  di baris same-`customer_id` konsisten dengan hipotesis ini).
- Tidak ada label negative genuine dari sumber ini sama sekali.

In [ ]:
cid_counts = df['customer_id'].value_counts()
dup_cids = cid_counts[cid_counts > 1].index

silver_pairs = []
for cid in dup_cids:
    idxs = df.index[df['customer_id'] == cid].tolist()
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            silver_pairs.append({
                'idx_l': idxs[i], 'idx_r': idxs[j],
                'customer_id': cid,
                'label': 'match',
                'label_source': 'silver_standard_customer_id'
            })

silver_df = pd.DataFrame(silver_pairs)
print(f"Silver-standard positive pairs: {len(silver_df):,}")

## 4. Sampling untuk manual review

Empat kategori sampel, masing-masing menutup celah berbeda di ground truth:

| Kategori | Tujuan | Sumber |
|---|---|---|
| A. Cluster size >= 3 | Deteksi over-merge (entity berbeda tergabung transitif) | `cluster_analysis.csv` |
| B. Pairs probability rendah (< 0.10) | Deteksi false negative (duplicate asli yang lolos dari model) | `decided_matches.csv` |
| C. Random sample dari decision=MATCH | Spot-check false positive di luar cluster besar | `decided_matches.csv` |
| D. Hard-negative candidates | Isi kekosongan genuine non-match yang sama sekali tidak ada di candidate set manapun | dibuat baru dari `df` |

**Kategori D paling penting** — ini yang akan mengatasi masalah saturasi probability
di iterasi berikutnya (dicatat sebagai limitasi di Notebook 04/05).

In [ ]:
# A. Semua anggota cluster size >= 3
large_cluster_ids = clusters.loc[clusters['cluster_size'] >= 3, 'cluster_id'].unique()
sample_a = clusters[clusters['cluster_id'].isin(large_cluster_ids)].copy()
sample_a['review_category'] = 'A_large_cluster'
print(f"[A] Anggota cluster size >= 3: {len(sample_a)} rows dari {len(large_cluster_ids)} cluster")

In [ ]:
# B. Pairs dengan probability < 0.10
sample_b = pred[pred['match_probability'] < 0.10].copy()
sample_b['review_category'] = 'B_low_probability'
print(f"[B] Pairs probability < 0.10: {len(sample_b)}")

In [ ]:
# C. Random sample dari decision=MATCH (di luar kategori A/B)
# Target: 100 pairs atau 10% dari total MATCH, mana yang lebih kecil
match_pairs = pred[pred['decision'] == 'MATCH'].copy()
n_sample_c = min(100, max(1, int(len(match_pairs) * 0.10)))
sample_c = match_pairs.sample(n=min(n_sample_c, len(match_pairs)), random_state=RANDOM_SEED).copy()
sample_c['review_category'] = 'C_random_match_spotcheck'
print(f"[C] Random spot-check dari MATCH: {len(sample_c)} dari {len(match_pairs)} total MATCH")

In [ ]:
# D. Hard-negative candidates: pasangan yang MIRIP di satu field tapi TIDAK ada
# di candidate_pairs manapun (customer_id BEDA, dan tidak satupun blocking key sama)
# Strategi: cari pasangan dengan last_name_std SAMA + city_std SAMA, tapi
# phone_main_std, email_std, dob_std SEMUA beda -- kandidat kuat genuine non-match
# (nama belakang+kota sama itu wajar terjadi pada orang berbeda, TIDAK seperti
# phone/email/dob yang lebih diskriminatif)

candidate_hard_neg = []
grouped = df.dropna(subset=['last_name_std', 'city_std']).groupby(['last_name_std', 'city_std'])
for (lname, city), group in grouped:
    if len(group) < 2:
        continue
    idxs = group.index.tolist()
    for i in range(len(idxs)):
        for j in range(i + 1, len(idxs)):
            r1, r2 = df.loc[idxs[i]], df.loc[idxs[j]]
            if r1['customer_id'] == r2['customer_id']:
                continue  # skip, ini sudah silver-standard
            diff_phone = r1.get('phone_main_std') != r2.get('phone_main_std')
            diff_email = r1.get('email_std') != r2.get('email_std')
            diff_dob   = r1.get('dob_std') != r2.get('dob_std')
            if diff_phone and diff_email and diff_dob:
                candidate_hard_neg.append({'idx_l': idxs[i], 'idx_r': idxs[j]})
    if len(candidate_hard_neg) > 5000:
        break

hard_neg_df = pd.DataFrame(candidate_hard_neg)
print(f"Kandidat hard-negative ditemukan: {len(hard_neg_df):,}")

n_sample_d = min(100, len(hard_neg_df))
sample_d = hard_neg_df.sample(n=n_sample_d, random_state=RANDOM_SEED).copy() if len(hard_neg_df) > 0 else hard_neg_df
sample_d['review_category'] = 'D_hard_negative_candidate'
print(f"[D] Hard-negative sample untuk review: {len(sample_d)}")

**Catatan penting soal kategori D:** ini BUKAN ground truth otomatis — ini kandidat yang
MASUK AKAL sebagai non-match berdasarkan heuristik (last_name+city sama tapi phone/email/dob
semua beda), tapi tetap WAJIB direview manual. Bisa saja ternyata memang orang yang sama
yang pindah nomor/email (jarang, tapi mungkin) — heuristik hanya mempersempit pencarian,
bukan memutuskan.

## 5. Gabungkan & export review queue

Semua kategori digabung jadi satu file dengan detail record lengkap (bukan cuma index),
supaya reviewer bisa bandingkan langsung tanpa buka file lain.

**Catatan teknis:** kategori A (cluster) butuh transformasi pair-wise dari cluster members
(self-join di dalam satu cluster_id), berbeda strukturnya dari B/C/D yang sudah dalam bentuk pair.

In [ ]:
def build_review_rows(sample, idx_l_col, idx_r_col, category_col='review_category'):
    rows = []
    display_cols = ['customer_id', 'first_name', 'last_name', 'email', 'phone_number',
                     'dob', 'address', 'city', 'country']
    for _, row in sample.iterrows():
        il, ir = int(row[idx_l_col]), int(row[idx_r_col])
        rl, rr = df.loc[il], df.loc[ir]
        entry = {'review_category': row[category_col]}
        for c in display_cols:
            entry[f'{c}_l'] = rl.get(c)
            entry[f'{c}_r'] = rr.get(c)
        rows.append(entry)
    return pd.DataFrame(rows)

# Kategori A: self-join dalam satu cluster_id untuk membentuk pairwise combinations
cluster_pairs = []
for cid, group in sample_a.groupby('cluster_id'):
    ids = group['unique_id'].astype(int).tolist()
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            cluster_pairs.append({
                'unique_id_l': ids[i], 'unique_id_r': ids[j],
                'review_category': 'A_large_cluster'
            })
sample_a_pairs = pd.DataFrame(cluster_pairs)
print(f"Kategori A dikonversi jadi {len(sample_a_pairs)} pairs")

In [ ]:
rows_a = build_review_rows(sample_a_pairs, 'unique_id_l', 'unique_id_r') if len(sample_a_pairs) > 0 else pd.DataFrame()
rows_b = build_review_rows(sample_b, 'unique_id_l', 'unique_id_r') if len(sample_b) > 0 else pd.DataFrame()
rows_c = build_review_rows(sample_c, 'unique_id_l', 'unique_id_r') if len(sample_c) > 0 else pd.DataFrame()
rows_d = build_review_rows(sample_d, 'idx_l', 'idx_r') if len(sample_d) > 0 else pd.DataFrame()

review_queue = pd.concat([rows_a, rows_b, rows_c, rows_d], ignore_index=True)
dedup_cols = [c for c in review_queue.columns if c != 'review_category']
review_queue = review_queue.drop_duplicates(subset=dedup_cols)
review_queue['manual_label'] = ''
review_queue['reviewer_notes'] = ''

print(f"Total review queue (setelah dedup): {len(review_queue):,} pairs")
print(review_queue['review_category'].value_counts())

review_queue.to_csv(REVIEW_QUEUE_PATH, index=False)
print(f"\nSaved: {REVIEW_QUEUE_PATH}")
print("\n>>> ACTION REQUIRED: buka file ini, isi kolom 'manual_label' untuk SEMUA baris,")
print(">>> simpan, lalu lanjutkan ke Section 6 di bawah.")

---
## STOP DI SINI -- Langkah Manual Diperlukan

Sebelum melanjutkan ke Section 6:

1. Buka `review_queue.csv` di Excel/spreadsheet
2. Untuk setiap baris, bandingkan kolom `_l` vs `_r`, isi `manual_label` dengan salah satu:
   - `match` -- yakin orang yang sama
   - `non-match` -- yakin orang berbeda
   - `unsure` -- tidak yakin (akan DIKELUARKAN dari ground truth, bukan dipaksa masuk)
3. Simpan file (format CSV, jangan diubah ke xlsx)
4. Lanjutkan ke cell di bawah

**Kenapa `unsure` dikeluarkan, bukan dipaksa jadi salah satu label:** ground truth yang
dipaksa dari kasus ambigu akan mencemari evaluasi Notebook 07 dengan noise. Lebih baik
ground truth lebih kecil tapi bersih, daripada besar tapi mengandung label yang tidak yakin.

## 6. Load hasil manual review, gabungkan dengan silver-standard, split tuning/holdout

In [ ]:
reviewed = pd.read_csv(REVIEW_QUEUE_PATH, dtype=str)

n_empty = (reviewed['manual_label'].isna() | (reviewed['manual_label'].str.strip() == '')).sum()
assert n_empty == 0, (
    f"Masih ada {n_empty} baris yang manual_label-nya kosong. "
    "Lengkapi dulu semua baris di review_queue.csv sebelum lanjut."
)

reviewed['manual_label'] = reviewed['manual_label'].str.strip().str.lower()
valid_labels = {'match', 'non-match', 'unsure'}
invalid = reviewed[~reviewed['manual_label'].isin(valid_labels)]
assert len(invalid) == 0, f"Ada label tidak valid: {invalid['manual_label'].unique()}"

print("Distribusi manual_label:")
print(reviewed['manual_label'].value_counts())

n_unsure = (reviewed['manual_label'] == 'unsure').sum()
print(f"\n{n_unsure} baris 'unsure' akan DIKELUARKAN dari ground truth final.")
reviewed_clean = reviewed[reviewed['manual_label'] != 'unsure'].copy()

In [ ]:
manual_gt = reviewed_clean[['review_category', 'manual_label']].copy()
manual_gt = manual_gt.rename(columns={'manual_label': 'label'})
manual_gt['label_source'] = 'manual_review'

silver_gt = silver_df[['label', 'label_source']].copy()
silver_gt['review_category'] = 'silver_standard'

ground_truth = pd.concat([silver_gt, manual_gt], ignore_index=True)

print("Komposisi ground truth final:")
print(ground_truth.groupby(['label_source', 'label']).size())
print(f"\nTotal ground truth: {len(ground_truth):,}")
print(f"  match     : {(ground_truth['label']=='match').sum():,}")
print(f"  non-match : {(ground_truth['label']=='non-match').sum():,}")

### Split tuning / holdout (Rule 6 -- Avoid data leakage)

70% tuning (dipakai untuk kalibrasi threshold di Notebook 07), 30% holdout
(dipakai HANYA untuk melaporkan angka final, tidak boleh dilihat sebelum threshold ditentukan).

Split dilakukan STRATIFIED per `label` supaya proporsi match/non-match seimbang di kedua set.

**Disclaimer wajib (sesuai Rule 6):** jika ukuran ground truth (khususnya non-match hasil
manual review) terlalu kecil untuk displit tanpa estimasi jadi tidak stabil, ini HARUS
dicatat sebagai limitasi eksplisit di Notebook 07 -- bukan disembunyikan di balik angka final.

In [ ]:
from math import floor

def stratified_split(gt_df, frac_tuning=0.7, seed=RANDOM_SEED):
    parts_tuning, parts_holdout = [], []
    for label, group in gt_df.groupby('label'):
        group = group.sample(frac=1.0, random_state=seed)
        n_tuning = floor(len(group) * frac_tuning)
        parts_tuning.append(group.iloc[:n_tuning])
        parts_holdout.append(group.iloc[n_tuning:])
    return pd.concat(parts_tuning), pd.concat(parts_holdout)

tuning_set, holdout_set = stratified_split(ground_truth)
tuning_set['split']  = 'tuning'
holdout_set['split'] = 'holdout'

print(f"Tuning set  : {len(tuning_set):,} ({(tuning_set['label']=='match').sum()} match, "
      f"{(tuning_set['label']=='non-match').sum()} non-match)")
print(f"Holdout set : {len(holdout_set):,} ({(holdout_set['label']=='match').sum()} match, "
      f"{(holdout_set['label']=='non-match').sum()} non-match)")

MIN_RECOMMENDED = 30
for name, s in [('tuning', tuning_set), ('holdout', holdout_set)]:
    n_nonmatch = (s['label']=='non-match').sum()
    if n_nonmatch < MIN_RECOMMENDED:
        print(f"\nWARNING: {name} set hanya punya {n_nonmatch} non-match label "
              f"(rekomendasi minimal {MIN_RECOMMENDED}). Estimasi precision/recall "
              f"di set ini BERISIKO TIDAK STABIL -- wajib dicatat sebagai limitasi di Notebook 07.")

In [ ]:
final_gt = pd.concat([tuning_set, holdout_set], ignore_index=True)
final_gt.to_csv(GROUND_TRUTH_PATH, index=False)
print(f"Saved: {GROUND_TRUTH_PATH} ({len(final_gt):,} rows)")

## Ringkasan Notebook 06

```text
Silver-standard positive pairs : <isi -- dari customer_id>
Review queue total              : <isi>
  - Kategori A (cluster >=3)    : <isi>
  - Kategori B (low probability): <isi>
  - Kategori C (random spot-check MATCH): <isi>
  - Kategori D (hard-negative candidate): <isi>
Manual review hasil             : <isi -- match / non-match / unsure count>
Ground truth final               : <isi -- total, tuning vs holdout>
Warning minimal non-match label  : <isi -- apakah terpenuhi>
```

**Limitasi yang harus dibawa ke Notebook 07:**
1. Silver-standard TIDAK divalidasi manual sepenuhnya -- based on asumsi customer_id.
2. Kategori D (hard-negative) adalah heuristik, bukan ground truth pasti sebelum direview.
3. Jika holdout non-match count di bawah rekomendasi, precision/recall final punya interval
   ketidakpastian besar -- harus dilaporkan dengan disclaimer, bukan angka tunggal yang pasti.

**Next:** `07_evaluation.ipynb` -- precision/recall/F1 pada tuning set (kalibrasi threshold)
dan holdout set (angka final).